# 🚗 [Notebook 2/2] Huấn Luyện Mô Hình & Đánh Giá Chi Tiết
**Repo:** [tunaleeZ/Driver_behaviour](https://github.com/tunaleeZ/Driver_behaviour)  
**Mục tiêu:** Sử dụng bộ dataset đã tiền xử lý từ Notebook 1 -> Huấn luyện mô hình Landmark-LSTM -> Vẽ đồ thị Loss/F1 & Ma trận nhầm lẫn -> Chạy suy luận trực tiếp trên video kiểm thử 60s và so sánh với Ground Truth -> Xuất video có gắn nhãn thời gian thực!

---
### ⚙️ Hướng dẫn cài đặt trên Kaggle:
1. **Accelerator:** Vào menu bên phải -> Chọn **GPU T4** (hoặc P100).
2. **Internet:** Vào menu bên phải -> Chọn **Internet On** (để clone mã nguồn và cài thư viện).
3. **Input Dataset:**
   - Bấm **+ Add Input** ở menu bên phải.
   - Chọn Dataset đã tạo từ Notebook 1 (ví dụ: `driver-behavior-preprocessed`) hoặc chọn trực tiếp Notebook Output của Notebook 1.


In [ ]:
# ==========================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ==========================================
import sys, os, subprocess
from pathlib import Path
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM khả dụng: {vram:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Notebook options -> Accelerator -> Chọn GPU T4 x2.")


## 2. Clone Repository & Cài Đặt Dependencies
Clone toàn bộ mã nguồn repo GitHub vào `/kaggle/working` và cài đặt gói dự án `dbm` cùng các thư viện bổ trợ.


In [ ]:
# ==========================================
# 2. CLONE GITHUB REPO & CÀI ĐẶT
# ==========================================
REPO_URL = "https://github.com/tunaleeZ/Driver_behaviour.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "Driver_behaviour"

if not REPO_DIR.exists():
    print(f"Đang clone repo từ {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo đã có sẵn tại {REPO_DIR}")

# Chuyển thư mục làm việc về REPO_DIR
os.chdir(REPO_DIR)
print("Thư mục làm việc hiện tại:", os.getcwd())

# Cài đặt các gói phụ thuộc bổ sung (Kaggle đã có sẵn torch, numpy, opencv)
print("Đang cài đặt thư viện cần thiết...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe>=0.10.0", "imageio-ffmpeg>=0.4.9", "openpyxl>=3.1.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)

# Thêm src vào sys.path
sys.path.insert(0, str(REPO_DIR / "src"))

import mediapipe as mp
import dbm
print("MediaPipe version:", mp.__version__)
print("Package dbm đã import thành công!")


## 3. Tự Động Kết Nối & Đồng Bộ Preprocessed Dataset
Tự động quét thư mục dataset đã tiền xử lý trong `/kaggle/input/` và liên kết vào thư mục làm việc của dự án.


In [ ]:
# ==========================================
# 3. ĐỒNG BỘ DỮ LIỆU ĐÃ TIỀN XỬ LÝ
# ==========================================
import os, shutil, zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_DIR = REPO_DIR / "driver_behavior_dataset_v01"
ANNOT_DIR = DATASET_DIR / "annotations"
LM_DIR = DATASET_DIR / "features" / "landmarks"
OUTPUTS_DIR = REPO_DIR / "outputs"

ANNOT_DIR.mkdir(parents=True, exist_ok=True)
LM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Tìm file zip hoặc thư mục chứa metadata.csv và landmarks
found_preprocessed = False
test_video_found = None
test_gt_found = None

for root, dirs, files in os.walk(KAGGLE_INPUT):
    root_path = Path(root)
    # Giải nén nếu là file zip
    for f in files:
        if f == "preprocessed_dataset.zip":
            print(f"Phát hiện file {f}, đang giải nén...")
            with zipfile.ZipFile(root_path / f, "r") as z:
                z.extractall(WORK_DIR / "unzipped_dataset")
            root_path = WORK_DIR / "unzipped_dataset"
            break

# Quét tìm kiếm dữ liệu sau giải nén / mount
search_roots = [WORK_DIR / "unzipped_dataset", KAGGLE_INPUT]
for s_root in search_roots:
    if not s_root.exists():
        continue
    for root, dirs, files in os.walk(s_root):
        p = Path(root)
        for f in files:
            if f in ["metadata.csv", "metadata.xlsx", "splits.csv", "inventory.csv"]:
                dst = ANNOT_DIR / f
                if not dst.exists():
                    shutil.copy2(p / f, dst)
                    found_preprocessed = True
            elif f.endswith(".npz"):
                dst = LM_DIR / f
                if not dst.exists():
                    shutil.copy2(p / f, dst)
                    found_preprocessed = True
            elif f == "test_unseen_clip.mp4":
                test_video_found = p / f
            elif f == "test_groundtruth.csv":
                test_gt_found = p / f

if test_video_found:
    TEST_VIDEO_PATH = WORK_DIR / "test_unseen_clip.mp4"
    if not TEST_VIDEO_PATH.exists():
        shutil.copy2(test_video_found, TEST_VIDEO_PATH)
    print(f"✓ Video test: {TEST_VIDEO_PATH.name} ({TEST_VIDEO_PATH.stat().st_size / 1e6:.2f} MB)")
else:
    TEST_VIDEO_PATH = WORK_DIR / "test_unseen_clip.mp4"
    print("⚠️ Không tìm thấy test_unseen_clip.mp4 trong input.")

if test_gt_found:
    shutil.copy2(test_gt_found, OUTPUTS_DIR / "test_groundtruth.csv")
    print(f"✓ Test Ground Truth: {OUTPUTS_DIR / "test_groundtruth.csv"}")

npz_count = len(list(LM_DIR.glob("*.npz")))
print(f"\nTổng kết dữ liệu sẵn sàng cho huấn luyện:")
print(f"  - Metadata: {ANNOT_DIR / \"metadata.csv\"} (Tồn tại: {(ANNOT_DIR / \"metadata.csv\").exists()})")
print(f"  - Splits:   {ANNOT_DIR / \"splits.csv\"} (Tồn tại: {(ANNOT_DIR / \"splits.csv\").exists()})")
print(f"  - Landmark features (.npz): {npz_count} files")

if npz_count == 0:
    raise RuntimeError("Chưa tìm thấy các file .npz đặc trưng! Hãy đảm bảo bạn đã Add đúng Dataset tiền xử lý từ Notebook 1.")


## 11. Huấn Luyện Mô Hình (Landmark-LSTM)
Huấn luyện mô hình chuỗi thời gian LSTM trên đặc trưng landmark với trọng số cân bằng lớp và early stopping.


In [ ]:
# ==========================================
# 11. HUẤN LUYỆN MODEL LANDMARK-LSTM
# ==========================================
CONFIG_PATH = "configs/models/landmark_lstm.yaml"
RUN_NAME = "landmark_lstm_run"

print(f"Bắt đầu huấn luyện mô hình với cấu hình: {CONFIG_PATH}...")
subprocess.run([
    sys.executable, "-m", "dbm.train",
    "--config", CONFIG_PATH,
    "--run-name", RUN_NAME,
    "--override", "batch_size=16",
    "--override", "max_epochs=25",
    "--override", "lr=0.001"
], check=True)


## 12. Đánh Giá & Vẽ Biểu Đồ (Loss Curves & Confusion Matrix)


In [ ]:
# ==========================================
# 12. ĐÁNH GIÁ VÀ HIỂN THỊ ĐỒ THỊ
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = REPO_DIR / "outputs" / RUN_NAME
best_ckpt = run_dir / "best.pt"
print(f"Đánh giá checkpoint tốt nhất: {best_ckpt}")

# Đánh giá lại trên tập val
subprocess.run([
    sys.executable, "-m", "dbm.evaluate",
    "--checkpoint", str(best_ckpt),
    "--split", "val"
], check=True)

# Hiển thị biểu đồ huấn luyện và ma trận nhầm lẫn
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

lc_img_path = run_dir / "learning_curves.png"
if lc_img_path.exists():
    img1 = mpimg.imread(str(lc_img_path))
    axs[0].imshow(img1)
    axs[0].set_title("Biểu Đồ Học Tập (Train/Val Loss & Macro-F1)", fontsize=14)
    axs[0].axis("off")

cm_img_path = run_dir / "confusion_matrix.png"
if cm_img_path.exists():
    img2 = mpimg.imread(str(cm_img_path))
    axs[1].imshow(img2)
    axs[1].set_title("Ma Trận Nhầm Lẫn (Validation Confusion Matrix)", fontsize=14)
    axs[1].axis("off")

plt.tight_layout()
plt.show()


## 13. Test Trực Tiếp Trên Video Test Độc Lập & Render Video Kết Quả
Cell này chạy suy luận trượt (Sliding-window Inference) trực tiếp trên file video test `test_unseen_clip.mp4` đã tách ở bước 5:
1. Xuất file bảng thống kê thời gian thực `outputs/test_timeline_predictions.csv` (thời điểm, nhãn dự đoán, xác suất từng lớp).
2. Vẽ nhãn dự đoán và thanh xác suất trực tiếp lên video và lưu file `outputs/test_annotated_video.mp4` để tải về xem trực quan!


In [ ]:
# ==========================================
# 13. TEST SUY LUẬN & SO SÁNH VỚI GROUND TRUTH
# ==========================================
csv_output = run_dir / "test_timeline_predictions.csv"
annotated_video_out = run_dir / "test_annotated_video.mp4"
test_gt_csv = REPO_DIR / "outputs" / "test_groundtruth.csv"

print(f"Đang suy luận trượt trên video: {TEST_VIDEO_PATH.name}...")
subprocess.run([
    sys.executable, "-m", "dbm.infer_video",
    "--video", str(TEST_VIDEO_PATH),
    "--checkpoint", str(best_ckpt),
    "--output", str(csv_output),
    "--smoothing", "ema"
], check=True)

df_preds = pd.read_csv(csv_output)
df_gt = pd.read_csv(test_gt_csv) if test_gt_csv.exists() else pd.DataFrame()

# Hàm lấy nhãn Ground Truth tại thời điểm t
def get_gt_label(t):
    if df_gt.empty:
        return "unknown"
    match = df_gt[(df_gt["test_clip_start"] <= t) & (df_gt["test_clip_end"] >= t)]
    return match.iloc[-1]["label"] if not match.empty else "normal"

# Thêm cột Ground Truth vào bảng dự đoán để so sánh
df_preds["mid_sec"] = (df_preds["start_seconds"] + df_preds["end_seconds"]) / 2.0
df_preds["ground_truth"] = df_preds["mid_sec"].apply(get_gt_label)
df_preds["is_correct"] = df_preds["prediction"] == df_preds["ground_truth"]
df_preds.to_csv(csv_output, index=False)

accuracy = df_preds["is_correct"].mean() * 100
print(f"\n✓ Đã đối chiếu với Ground Truth: Độ chính xác trên đoạn test 60s = {accuracy:.2f}%")
print("\nBảng so sánh Ground Truth vs Prediction (10 cửa sổ đầu tiên):")
print(df_preds[["start_seconds", "end_seconds", "ground_truth", "prediction", "is_correct"]].head(10))

# --- RENDER VIDEO HIỂN THỊ CẢ GROUND TRUTH VÀ PREDICTION ---
print(f"\nĐang vẽ Ground Truth & Prediction lên video -> {annotated_video_out.name}...")
cap = cv2.VideoCapture(str(TEST_VIDEO_PATH))
v_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(annotated_video_out), cv2.VideoWriter_fourcc(*"mp4v"), v_fps, (width, height))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    current_sec = frame_idx / v_fps
    
    # Lấy prediction và GT tại frame hiện tại
    active_pred = df_preds[(df_preds["start_seconds"] <= current_sec) & (df_preds["end_seconds"] >= current_sec)]
    pred_label = active_pred.iloc[-1]["prediction"] if not active_pred.empty else "analyzing..."
    gt_label = get_gt_label(current_sec)
    
    is_match = (pred_label == gt_label)
    status_text = "CORRECT" if is_match else "MISMATCH"
    status_color = (0, 255, 0) if is_match else (0, 0, 255)  # Xanh nếu đúng, Đỏ nếu sai
    
    # Vẽ HUD hiển thị thông tin
    cv2.rectangle(frame, (20, 20), (520, 130), (0, 0, 0), -1)
    cv2.putText(frame, f"Time: {current_sec:.1f}s / {TEST_DURATION_SEC}s", (35, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(frame, f"GT   (Thuc te): {gt_label.upper()}", (35, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(frame, f"PRED (Du doan): {pred_label.upper()} [{status_text}]", (35, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, status_color, 2)
    
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"✓ Đã tạo video so sánh trực quan thành công: {annotated_video_out.name} ({annotated_video_out.stat().st_size / 1e6:.2f} MB)")


## 14. Nén Toàn Bộ Kết Quả & Video Test Để Tải Về


In [ ]:
# ==========================================
# 14. NÉN KẾT QUẢ, MODEL & VIDEO TEST ĐỂ TẢI VỀ
# ==========================================
zip_out = WORK_DIR / "driver_behaviour_results.zip"
print(f"Đang nén toàn bộ outputs và video test vào {zip_out}...")
subprocess.run(["zip", "-r", str(zip_out), "outputs/"], check=True)
print(f" Hoàn tất! Bạn có thể tải file '{zip_out.name}' trực tiếp tại panel Output bên phải của Kaggle.")
